# 04) Deployment Visualizations

**SDC Master's Final Project: soccernet-setpiece-vision**

**CRISP-DM phase 6 (Deployment).** A Pitch Control number is only useful if a practitioner can read it. This notebook turns the validated pipeline output into the form an analyst actually consumes: a tactical view of each set piece that places players and control on the pitch without requiring any data-science background. This is the project's deliverable in the hands of its intended user (a club or scout with broadcast footage and a laptop).

The headline output is a three-panel view per set piece:

| Panel | Shows |
|---|---|
| Broadcast frame + detections | Where the pipeline found each player and the referee, on the original footage |
| Metric minimap | The same players as a clean top-down picture, coloured by team, with the ball |
| Pitch Control heatmap | Which team controls which space (attacker control in red, defender in blue) |

These are rendered as annotated stills, animated GIFs, and MP4 clips for the representative corner and direct free kick, plus a standalone detection figure that documents the detector's raw output. A final section reproduces the minimap and Pitch Control views directly from stored coordinates, so the core deployment picture can be regenerated from the project's published data alone.

*Which metrics are trustworthy enough to put in front of a practitioner is established in notebook 03; this notebook is about presenting them well.*

## 0. Setup

In [1]:
import json
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle
from mplsoccer import Pitch

PROJECT_ROOT = Path.cwd().parent
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
GSR_ROOT = Path(os.getenv("SOCCERNET_LOCAL_DIR", "/Volumes/MPH-ExternalStorage/soccernet-gsr")) / "gamestate-2024"

PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0

# PC params (must match nb03 exactly)
GRID_NX, GRID_NY = 60, 40
MAX_SPEED = 5.0
REACTION_TIME = 0.7
SIGMA = 0.45
TIME_TO_INTERCEPT_SIGMOID_K = float(np.pi / (np.sqrt(3.0) * SIGMA))

# Colours
TEAM_COLOURS = {0: "#1f77b4", 1: "#d62728", -1: "#888888"}
REFEREE_COLOUR = "#ff7f0e"
BALL_COLOUR = "#000000"

print("OK")

OK


## 1. Pitch Control model

The same Pitch Control model used in the evaluation, restated here so the visualizations show exactly the surface that was validated not a re-parameterised variant. Identical constants guarantee the heatmaps a practitioner sees match the numbers in notebook 03.

In [2]:
def time_to_intercept(player_xy: np.ndarray, target_xy: np.ndarray) -> np.ndarray:
    if len(player_xy) == 0:
        return np.full((1, len(target_xy)), np.inf)
    d = np.linalg.norm(player_xy[:, None, :] - target_xy[None, :, :], axis=2)
    return REACTION_TIME + d / MAX_SPEED


def pitch_control_surface(att_xy: np.ndarray, def_xy: np.ndarray) -> np.ndarray:
    xs = np.linspace(0.0, PITCH_LENGTH_M, GRID_NX)
    ys = np.linspace(0.0, PITCH_WIDTH_M, GRID_NY)
    grid = np.array(np.meshgrid(xs, ys)).reshape(2, -1).T
    if len(att_xy) == 0 or len(def_xy) == 0:
        return np.full((GRID_NY, GRID_NX), 0.5)
    tti_att = time_to_intercept(att_xy, grid).min(axis=0)
    tti_def = time_to_intercept(def_xy, grid).min(axis=0)
    delta = tti_att - tti_def
    p_att = 1.0 / (1.0 + np.exp(TIME_TO_INTERCEPT_SIGMOID_K * delta))
    return p_att.reshape(GRID_NY, GRID_NX)


def split_attack_defend(players_xy: np.ndarray, team_labels: np.ndarray, ball_xy: tuple):
    bx, by = ball_xy
    teams = [t for t in np.unique(team_labels) if t != -1 and not np.isnan(t)]
    if len(teams) < 2:
        return players_xy, np.zeros((0, 2)), teams[0] if teams else None
    min_d = {}
    for t in teams:
        mask = team_labels == t
        if mask.sum() == 0:
            continue
        d = np.linalg.norm(players_xy[mask] - np.array([bx, by]), axis=1)
        min_d[t] = d.min()
    att = min(min_d.keys(), key=lambda k: min_d[k])
    deff = next(t for t in min_d if t != att)
    return players_xy[team_labels == att], players_xy[team_labels == deff], att


print("PC functions ready")

PC functions ready


## 2. Load player positions and detections

Load the pipeline's per-frame player detections and positions, the input every panel draws on. The broadcast overlays in Sections 4–8 additionally place these on the original video frames; Section 9 needs only the coordinates loaded here.

In [3]:
# This notebook renders broadcast overlays: every panel needs the pipeline Soccana detections
# (pixel boxes + team labels) AND the broadcast JPEGs on the SSD. The broadcast video is never
# redistributed, so without the SSD frames there is nothing to draw. We detect that here and set
# CAN_RENDER; the render cells below skip cleanly so the notebook runs top-to-bottom.
PIPE_DET_PATH = OUTPUTS_DIR / "detections_soccana_tvcalib.parquet"
CAN_RENDER = PIPE_DET_PATH.is_file() and GSR_ROOT.is_dir()

gt_df = pd.read_parquet(OUTPUTS_DIR / "detections_gt_full.parquet")
if PIPE_DET_PATH.is_file():
    pipe_df = pd.read_parquet(PIPE_DET_PATH)
else:
    pipe_df = None


def parse_ball_position(clip_path: Path, frame_idx: int):
    label_path = clip_path / "Labels-GameState.json"
    if not label_path.is_file():
        return None
    with open(label_path) as f:
        labels = json.load(f)
    target = f"{frame_idx:06d}.jpg"
    image_id = next((img["image_id"] for img in labels["images"] if img.get("file_name") == target), None)
    if image_id is None:
        return None
    for a in labels["annotations"]:
        if a.get("image_id") != image_id or a.get("category_id") != 4:
            continue
        bp = a.get("bbox_pitch")
        if not bp:
            continue
        x = bp.get("x_bottom_middle")
        y = bp.get("y_bottom_middle")
        if x is None or y is None:
            continue
        return float(x) + PITCH_LENGTH_M / 2, float(y) + PITCH_WIDTH_M / 2
    return None


def parse_bbox_image(clip_path: Path, frame_idx: int):
    """Return list of {'bbox': (x,y,w,h), 'role': str} from GT JSON for overlaying GT."""
    label_path = clip_path / "Labels-GameState.json"
    if not label_path.is_file():
        return []
    with open(label_path) as f:
        labels = json.load(f)
    target = f"{frame_idx:06d}.jpg"
    image_id = next((img["image_id"] for img in labels["images"] if img.get("file_name") == target), None)
    if image_id is None:
        return []
    out = []
    for a in labels["annotations"]:
        if a.get("image_id") != image_id:
            continue
        b = a.get("bbox_image")
        if not b:
            continue
        out.append({"x": b.get("x"), "y": b.get("y"),
                    "w": b.get("w"), "h": b.get("h"),
                    "category": a.get("category_id")})
    return out


if CAN_RENDER:
    print(f"pipeline: {pipe_df.shape}, gt: {gt_df.shape}")
    print("clips with detections:", pipe_df['clip_id'].nunique())
else:
    missing = []
    if not PIPE_DET_PATH.is_file():
        missing.append("pipeline detections_soccana_tvcalib.parquet")
    if not GSR_ROOT.is_dir():
        missing.append(f"SSD broadcast frames ({GSR_ROOT})")
    print("CAN_RENDER = False — missing:", ", ".join(missing))
    print("This notebook produces broadcast overlays only; the render cells below will skip.")

CAN_RENDER = False — missing: pipeline detections_soccana_tvcalib.parquet, SSD broadcast frames (\Volumes\MPH-ExternalStorage\soccernet-gsr\gamestate-2024)
This notebook produces broadcast overlays only; the render cells below will skip.


## 3. Choose representative clips

We pick one corner and one direct free kick to illustrate, choosing the clips with the fullest frame coverage so the animations are smooth and complete. A few clips whose annotated set-piece type does not match what the footage actually shows are excluded, so the examples are honest.

In [4]:
# Clips whose SoccerNet action_class annotation does not match the visible content.
# SNGS-125: annotated as Corner but shows a mid-game scene.
# SNGS-131: annotated as Direct free-kick but shows a throw-in.
EXCLUDE_CLIPS = {"SNGS-125", "SNGS-131"}

if CAN_RENDER:
    frame_counts = (pipe_df.groupby(["split", "clip_id", "action_class"])["frame_idx"]
                    .nunique().reset_index(name="n_frames"))
    frame_counts = frame_counts[~frame_counts["clip_id"].isin(EXCLUDE_CLIPS)]
    best = (frame_counts.sort_values(["action_class", "n_frames"], ascending=[True, False])
                        .groupby("action_class").head(1))
    print(best.to_string(index=False))
else:
    best = pd.DataFrame(columns=["split", "clip_id", "action_class", "n_frames"])
    print("CAN_RENDER = False — no representative clips selected.")

CAN_RENDER = False — no representative clips selected.


## 4. Animation builder

The routine that assembles one animated set-piece view: for each frame it draws the broadcast image with detection boxes, the team-coloured minimap, and the Pitch Control heatmap, then stitches the frames into a clip. This is the engine behind the GIF and MP4 outputs.

In [5]:
def overlay_bboxes(ax, f_sub: pd.DataFrame) -> None:
    """Draw team-colored player boxes and referee boxes (orange) from parquet pixel coords."""
    has_referee_col = "is_referee" in f_sub.columns
    for _, r in f_sub.iterrows():
        x1, y1 = r["x1_px"], r["y1_px"]
        w = r["x2_px"] - x1
        h = r["y2_px"] - y1
        is_ref = bool(r["is_referee"]) if has_referee_col else False
        color = REFEREE_COLOUR if is_ref else TEAM_COLOURS.get(int(r["team_kmeans"]), "#888888")
        ax.add_patch(Rectangle((x1, y1), w, h, linewidth=1.5, edgecolor=color, facecolor="none"))


print("overlay_bboxes ready")

overlay_bboxes ready


In [6]:
def animate_clip(split: str, clip_id: str, action_class: str, fps: int = 6, out_path: Path | None = None):
    clip_path = GSR_ROOT / split / clip_id
    sub = pipe_df[(pipe_df["split"] == split) & (pipe_df["clip_id"] == clip_id)].copy()
    frames = sorted(sub["frame_idx"].unique())
    if not frames:
        print(f"  no frames for {clip_id}")
        return None

    fig = plt.figure(figsize=(16, 5))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 1.0])
    ax_bcast = fig.add_subplot(gs[0])
    ax_mini = fig.add_subplot(gs[1])
    ax_pc = fig.add_subplot(gs[2])
    pitch = Pitch(pitch_type="custom", pitch_length=PITCH_LENGTH_M, pitch_width=PITCH_WIDTH_M,
                  line_color="#444", pitch_color="#fafafa")

    def draw(frame_idx: int):
        ax_bcast.clear(); ax_mini.clear(); ax_pc.clear()

        # --- broadcast + detection boxes ---
        img_path = clip_path / "img1" / f"{frame_idx:06d}.jpg"
        if img_path.is_file():
            raw = cv2.imread(str(img_path))
            if raw is not None:
                ax_bcast.imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
        ax_bcast.set_xticks([]); ax_bcast.set_yticks([])
        ax_bcast.set_title(f"Broadcast + detections, frame {frame_idx}")

        # --- frame data ---
        f_sub = sub[sub["frame_idx"] == frame_idx]
        ball_xy = parse_ball_position(clip_path, frame_idx)

        overlay_bboxes(ax_bcast, f_sub)

        # --- minimap ---
        pitch.draw(ax=ax_mini)
        players_sub = f_sub[~f_sub["is_referee"]] if "is_referee" in f_sub.columns else f_sub
        refs_sub = f_sub[f_sub["is_referee"]] if "is_referee" in f_sub.columns else f_sub.iloc[0:0]
        for team_lbl, c in TEAM_COLOURS.items():
            mask = players_sub["team_kmeans"] == team_lbl
            if mask.any():
                ax_mini.scatter(players_sub.loc[mask, "x_m"], players_sub.loc[mask, "y_m"],
                                s=70, c=c, edgecolors="black", linewidths=0.6, zorder=3)
        if not refs_sub.empty:
            ax_mini.scatter(refs_sub["x_m"], refs_sub["y_m"],
                            s=60, c=REFEREE_COLOUR, marker="^",
                            edgecolors="black", linewidths=0.6, zorder=3)
        if ball_xy is not None:
            ax_mini.scatter([ball_xy[0]], [ball_xy[1]], s=80, c=BALL_COLOUR,
                            marker="*", edgecolors="white", linewidths=0.8, zorder=4)
        ax_mini.set_title("Minimap (metric pitch)")

        # --- PC ---
        pitch.draw(ax=ax_pc)
        if ball_xy is not None and not players_sub.empty:
            players = players_sub[["x_m", "y_m"]].to_numpy()
            teams = players_sub["team_kmeans"].to_numpy()
            att, deff, _ = split_attack_defend(players, teams, ball_xy)
            pc = pitch_control_surface(att, deff)
            extent = (0, PITCH_LENGTH_M, 0, PITCH_WIDTH_M)
            ax_pc.imshow(pc, origin="lower", extent=extent, cmap="RdBu_r",
                         vmin=0, vmax=1, alpha=0.65, zorder=2)
            ax_pc.scatter([ball_xy[0]], [ball_xy[1]], s=60, c=BALL_COLOUR,
                          marker="*", edgecolors="white", zorder=5)
        ax_pc.set_title("Pitch Control (attacker p)")

        fig.suptitle(f"{action_class}: {clip_id}", fontsize=12)

    def update(i):
        draw(frames[i])
        return []

    anim = FuncAnimation(fig, update, frames=len(frames), interval=1000 // fps, blit=False)
    if out_path is None:
        out_path = FIGURES_DIR / f"anim_{action_class.replace(' ', '_').lower()}_{clip_id}.gif"
    anim.save(out_path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f"  saved {out_path.name}  ({len(frames)} frames, {fps} fps)")
    return out_path

## 5. Render the animated clips

Produce the animated GIF for each chosen set piece. The moving version of the three-panel view, showing how control evolves across the set-piece window.

In [7]:
saved = []
for _, row in best.iterrows():
    p = animate_clip(row["split"], row["clip_id"], row["action_class"], fps=6)
    if p is not None:
        saved.append(p)

print("\nDone:")
for p in saved:
    print(f"  {p.relative_to(PROJECT_ROOT)}")


Done:


## 6. Static three-panel still

A single frame from each chosen clip, captured as a high-resolution still for embedding in the thesis. It freezes the most representative moment of the set piece into one publishable figure.

In [8]:
def render_still(split: str, clip_id: str, action_class: str, frame_idx: int):
    clip_path = GSR_ROOT / split / clip_id
    sub = pipe_df[(pipe_df["split"] == split) & (pipe_df["clip_id"] == clip_id) &
                  (pipe_df["frame_idx"] == frame_idx)]
    fig = plt.figure(figsize=(16, 5))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 1.0])
    ax_bcast = fig.add_subplot(gs[0])
    ax_mini = fig.add_subplot(gs[1])
    ax_pc = fig.add_subplot(gs[2])
    pitch = Pitch(pitch_type="custom", pitch_length=PITCH_LENGTH_M, pitch_width=PITCH_WIDTH_M,
                  line_color="#444", pitch_color="#fafafa")

    img_path = clip_path / "img1" / f"{frame_idx:06d}.jpg"
    if img_path.is_file():
        raw = cv2.imread(str(img_path))
        if raw is not None:
            ax_bcast.imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
    overlay_bboxes(ax_bcast, sub)
    ax_bcast.set_xticks([]); ax_bcast.set_yticks([])
    ax_bcast.set_title(f"Broadcast + detections (frame {frame_idx})")

    players_sub = sub[~sub["is_referee"]] if "is_referee" in sub.columns else sub
    refs_sub = sub[sub["is_referee"]] if "is_referee" in sub.columns else sub.iloc[0:0]

    ball_xy = parse_ball_position(clip_path, frame_idx)
    pitch.draw(ax=ax_mini)
    for team_lbl, c in TEAM_COLOURS.items():
        mask = players_sub["team_kmeans"] == team_lbl
        if mask.any():
            ax_mini.scatter(players_sub.loc[mask, "x_m"], players_sub.loc[mask, "y_m"],
                            s=70, c=c, edgecolors="black", linewidths=0.6, zorder=3)
    if not refs_sub.empty:
        ax_mini.scatter(refs_sub["x_m"], refs_sub["y_m"],
                        s=60, c=REFEREE_COLOUR, marker="^",
                        edgecolors="black", linewidths=0.6, zorder=3)
    if ball_xy is not None:
        ax_mini.scatter([ball_xy[0]], [ball_xy[1]], s=80, c=BALL_COLOUR,
                        marker="*", edgecolors="white", linewidths=0.8, zorder=4)
    ax_mini.set_title("Minimap")

    pitch.draw(ax=ax_pc)
    if ball_xy is not None and not players_sub.empty:
        players = players_sub[["x_m", "y_m"]].to_numpy()
        teams = players_sub["team_kmeans"].to_numpy()
        att, deff, _ = split_attack_defend(players, teams, ball_xy)
        pc = pitch_control_surface(att, deff)
        ax_pc.imshow(pc, origin="lower", extent=(0, PITCH_LENGTH_M, 0, PITCH_WIDTH_M),
                     cmap="RdBu_r", vmin=0, vmax=1, alpha=0.65, zorder=2)
        ax_pc.scatter([ball_xy[0]], [ball_xy[1]], s=60, c=BALL_COLOUR,
                      marker="*", edgecolors="white", zorder=5)
    ax_pc.set_title("Pitch Control")

    fig.suptitle(f"{action_class}: {clip_id} (frame {frame_idx})", fontsize=12)
    out_path = FIGURES_DIR / f"still_{action_class.replace(' ', '_').lower()}_{clip_id}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


for _, row in best.iterrows():
    sub = pipe_df[(pipe_df["split"] == row["split"]) & (pipe_df["clip_id"] == row["clip_id"])]
    centre = int(np.median(sub["frame_idx"].unique()))
    p = render_still(row["split"], row["clip_id"], row["action_class"], centre)
    print(f"  {p.name}")

### Outputs of Sections 5–6

- Animated three-panel GIFs for the representative corner and direct free kick. Broadcast frame with detections, minimap, and Pitch Control heatmap evolving across the set-piece window.
- A static three-panel still per clip (the median frame), for thesis embedding.

Together these are the moving and frozen versions of the same tactical view.

## 7. MP4 video export

The same three-panel view exported as an MP4, the format most convenient for sharing in a presentation or embedding in a talk. Content is identical to the GIF; only the container differs.

In [9]:
import io


def export_mp4(
    split: str,
    clip_id: str,
    action_class: str,
    fps: int = 6,
    out_path: Path | None = None,
) -> Path | None:
    clip_path = GSR_ROOT / split / clip_id
    sub = pipe_df[(pipe_df["split"] == split) & (pipe_df["clip_id"] == clip_id)].copy()
    frames = sorted(sub["frame_idx"].unique())
    if not frames:
        print(f"  no pipeline frames for {clip_id} - skipping")
        return None

    first_img_path = clip_path / "img1" / f"{frames[0]:06d}.jpg"
    if not first_img_path.is_file():
        print(f"  broadcast frames not found at {clip_path} - SSD mounted?")
        return None

    if out_path is None:
        label = action_class.replace(" ", "_").lower()
        out_path = FIGURES_DIR / f"video_{label}_{clip_id}.mp4"

    fig = plt.figure(figsize=(16, 5), dpi=120)
    gs_layout = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 1.0])
    ax_bcast = fig.add_subplot(gs_layout[0])
    ax_mini = fig.add_subplot(gs_layout[1])
    ax_pc = fig.add_subplot(gs_layout[2])
    pitch = Pitch(
        pitch_type="custom",
        pitch_length=PITCH_LENGTH_M,
        pitch_width=PITCH_WIDTH_M,
        line_color="#444",
        pitch_color="#fafafa",
    )

    def render_frame(frame_idx: int) -> np.ndarray:
        ax_bcast.clear()
        ax_mini.clear()
        ax_pc.clear()

        img_path = clip_path / "img1" / f"{frame_idx:06d}.jpg"
        if img_path.is_file():
            raw = cv2.imread(str(img_path))
            if raw is not None:
                ax_bcast.imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
        ax_bcast.set_xticks([])
        ax_bcast.set_yticks([])
        ax_bcast.set_title(f"Broadcast + detections  frame {frame_idx}")

        f_sub = sub[sub["frame_idx"] == frame_idx]
        ball_xy = parse_ball_position(clip_path, frame_idx)

        overlay_bboxes(ax_bcast, f_sub)

        players_sub = f_sub[~f_sub["is_referee"]] if "is_referee" in f_sub.columns else f_sub
        refs_sub = f_sub[f_sub["is_referee"]] if "is_referee" in f_sub.columns else f_sub.iloc[0:0]

        pitch.draw(ax=ax_mini)
        for team_lbl, colour in TEAM_COLOURS.items():
            mask = players_sub["team_kmeans"] == team_lbl
            if mask.any():
                ax_mini.scatter(
                    players_sub.loc[mask, "x_m"], players_sub.loc[mask, "y_m"],
                    s=70, c=colour, edgecolors="black", linewidths=0.6, zorder=3,
                )
        if not refs_sub.empty:
            ax_mini.scatter(
                refs_sub["x_m"], refs_sub["y_m"],
                s=60, c=REFEREE_COLOUR, marker="^", edgecolors="black", linewidths=0.6, zorder=3,
            )
        if ball_xy is not None:
            ax_mini.scatter(
                [ball_xy[0]], [ball_xy[1]],
                s=80, c=BALL_COLOUR, marker="*", edgecolors="white", linewidths=0.8, zorder=4,
            )
        ax_mini.set_title("Minimap (metric pitch)")

        pitch.draw(ax=ax_pc)
        if ball_xy is not None and not players_sub.empty:
            players = players_sub[["x_m", "y_m"]].to_numpy()
            teams = players_sub["team_kmeans"].to_numpy()
            att, deff, _ = split_attack_defend(players, teams, ball_xy)
            pc = pitch_control_surface(att, deff)
            ax_pc.imshow(
                pc, origin="lower",
                extent=(0, PITCH_LENGTH_M, 0, PITCH_WIDTH_M),
                cmap="RdBu_r", vmin=0, vmax=1, alpha=0.65, zorder=2,
            )
            ax_pc.scatter(
                [ball_xy[0]], [ball_xy[1]],
                s=60, c=BALL_COLOUR, marker="*", edgecolors="white", zorder=5,
            )
        ax_pc.set_title("Pitch Control (attacker p)")

        fig.suptitle(f"{action_class}: {clip_id}", fontsize=12)
        fig.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=120)
        buf.seek(0)
        arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
        bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        buf.close()
        assert bgr is not None
        return bgr

    first_frame = render_frame(frames[0])
    h, w = first_frame.shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # type: ignore[attr-defined]
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    writer.write(first_frame)
    for frame_idx in frames[1:]:
        writer.write(render_frame(frame_idx))
    writer.release()
    plt.close(fig)
    print(f"  saved {out_path.name}  ({len(frames)} frames, {fps} fps, {w}x{h})")
    return out_path


saved_mp4 = []
for _, row in best.iterrows():
    p = export_mp4(row["split"], row["clip_id"], row["action_class"], fps=6)
    if p is not None:
        saved_mp4.append(p)

print("\nMP4 outputs:")
for p in saved_mp4:
    print(f"  {p.relative_to(PROJECT_ROOT)}")


MP4 outputs:


## 8. Multi-class detection figure

A standalone look at the detector itself: Soccana's raw Player / Ball / Referee boxes with confidence scores, drawn on the single frame with the most player detections. This documents the quality of the detection stage that everything downstream depends on, and is the source for the detector figure in the thesis methods section.

In [10]:
if not CAN_RENDER:
    soccana_model = None
    print("CAN_RENDER = False — skipping Soccana multi-class detection figure (needs SSD + pipeline detections).")
else:
    import torch

    _orig_torch_load = torch.load
    torch.load = lambda *a, **kw: _orig_torch_load(*a, **{**kw, "weights_only": False})  # type: ignore[assignment]

    from huggingface_hub import hf_hub_download
    from ultralytics import YOLO

    SOCCANA_REPO = "Adit-jain/soccana"
    SOCCANA_WEIGHTS = "Model/weights/best.pt"
    DEVICE = os.getenv("TORCH_DEVICE", "mps")
    SOCCANA_CONF = 0.40

    CLASS_COLORS_BGR = {0: (50, 205, 50), 1: (0, 215, 255), 2: (0, 80, 220)}
    CLASS_NAMES_SOCCANA = {0: "Player", 1: "Ball", 2: "Referee"}

    # --- pick frame with most player detections ---
    by_frame = (
        pipe_df[~pipe_df["is_referee"]]
        .groupby(["split", "clip_id", "frame_idx"])
        .size()
        .reset_index(name="n")
        .sort_values("n", ascending=False)
    )
    best_row = by_frame.iloc[0]
    mc_split, mc_clip, mc_frame = str(best_row["split"]), str(best_row["clip_id"]), int(best_row["frame_idx"])
    print(f"Best frame: {mc_split}/{mc_clip} frame {mc_frame}  ({int(best_row['n'])} players in parquet)")

    mc_img_path = GSR_ROOT / mc_split / mc_clip / "img1" / f"{mc_frame:06d}.jpg"
    mc_img_bgr = cv2.imread(str(mc_img_path))

    print("Loading Soccana weights...")
    soccana_weights = hf_hub_download(repo_id=SOCCANA_REPO, filename=SOCCANA_WEIGHTS)
    soccana_model = YOLO(soccana_weights)
    print("Model ready")

CAN_RENDER = False — skipping Soccana multi-class detection figure (needs SSD + pipeline detections).


In [11]:
if not CAN_RENDER or soccana_model is None:
    print("CAN_RENDER = False — skipping (no Soccana model / broadcast frame).")
else:
    results = soccana_model.predict(
        source=mc_img_bgr,
        conf=SOCCANA_CONF,
        classes=[0, 1, 2],
        device=DEVICE,
        verbose=False,
    )[0]

    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    confs = results.boxes.conf.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    counts = {CLASS_NAMES_SOCCANA[c]: int((classes == c).sum()) for c in [0, 1, 2]}
    print(f"Detections at conf≥{SOCCANA_CONF}: {counts}")

    # draw boxes on copy
    annotated = mc_img_bgr.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX
    for (x1, y1, x2, y2), conf, cls in zip(boxes_xyxy, confs, classes):
        cls = int(cls)
        color = CLASS_COLORS_BGR.get(cls, (200, 200, 200))
        thickness = 3 if cls == 1 else 2
        cv2.rectangle(annotated, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)
        label = f"{CLASS_NAMES_SOCCANA.get(cls, str(cls))} {conf:.2f}"
        (tw, th), baseline = cv2.getTextSize(label, font, 0.55, 1)
        ty = max(int(y1) - 4, th + 2)
        cv2.rectangle(annotated, (int(x1), ty - th - baseline), (int(x1) + tw + 2, ty + baseline), color, -1)
        cv2.putText(annotated, label, (int(x1) + 1, ty), font, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(annotated_rgb)
    ax.axis("off")
    ax.set_title(
        f"Soccana (YOLOv11n) — {mc_clip} frame {mc_frame:06d} | conf≥{SOCCANA_CONF}",
        fontsize=13, pad=10,
    )
    legend_handles = [
        mpatches.Patch(
            color=tuple(c / 255 for c in CLASS_COLORS_BGR[cls][::-1]),
            label=f"{CLASS_NAMES_SOCCANA[cls]} ({counts[CLASS_NAMES_SOCCANA[cls]]})",
        )
        for cls in [0, 2, 1]
    ]
    ax.legend(handles=legend_handles, loc="upper left", fontsize=11, framealpha=0.8, edgecolor="white")

    fig11_path = FIGURES_DIR / "11_multiclass_detections.png"
    fig.savefig(fig11_path, dpi=150, bbox_inches="tight", facecolor="black")
    plt.show()
    plt.close(fig)
    print(f"Saved → {fig11_path.relative_to(PROJECT_ROOT)}")

CAN_RENDER = False — skipping (no Soccana model / broadcast frame).


## 9. Tactical view from coordinates alone

The two analytically meaningful panels (the metric minimap and the Pitch Control heatmap) depend only on player coordinates, not on the broadcast image. This section renders them directly from the stored positions, producing the same top-down tactical picture a practitioner reads, reproducible from the project's published data without the original video.

It shows both sources side by side where available: ground truth (always) and the pipeline (when its detections are present), making the deployment view a like-for-like comparison of what the pipeline delivers against the annotated reference.

Output: `outputs/figures/deploy_minimap_pc_ssd_free_<source>_<clip>.png`.

In [ ]:
# --- SSD-free ball lookup from committed PC parquets ---
_pc_parts = []
for _name in ["pitch_control_gt_full.parquet", "pitch_control_soccana_tvcalib.parquet"]:
    _p = OUTPUTS_DIR / _name
    if _p.is_file():
        _pc_parts.append(pd.read_parquet(_p)[["split", "clip_id", "frame_idx", "ball_x_m", "ball_y_m"]])
ball_lookup = (
    pd.concat(_pc_parts, ignore_index=True)
    .dropna(subset=["ball_x_m", "ball_y_m"])
    .drop_duplicates(subset=["split", "clip_id", "frame_idx"])
    .set_index(["split", "clip_id", "frame_idx"])
    if _pc_parts else pd.DataFrame()
)


def _ball_xy(split, clip_id, frame_idx):
    try:
        r = ball_lookup.loc[(split, clip_id, int(frame_idx))]
        return float(r["ball_x_m"]), float(r["ball_y_m"])
    except KeyError:
        return None


def _normalise_team(series: pd.Series) -> np.ndarray:
    """Map team labels to {0, 1, -1} for both GT ('left'/'right') and pipeline (0/1) sources."""
    vals = series.tolist()
    out = []
    for v in vals:
        if v in ("left", 0, 0.0, "0"):
            out.append(0)
        elif v in ("right", 1, 1.0, "1"):
            out.append(1)
        else:
            out.append(-1)  # referee / unknown
    return np.array(out, dtype=float)


def render_minimap_pc(det_df, team_col, split, clip_id, frame_idx, source_label):
    """2-panel (minimap + PC) for one frame, from coordinates only. Returns (fig, ok)."""
    g = det_df[(det_df["split"] == split) & (det_df["clip_id"] == clip_id) &
               (det_df["frame_idx"] == frame_idx)]
    ball_xy = _ball_xy(split, clip_id, frame_idx)
    fig, (ax_mini, ax_pc) = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
    pitch = Pitch(pitch_type="custom", pitch_length=PITCH_LENGTH_M, pitch_width=PITCH_WIDTH_M,
                  line_color="#444", pitch_color="#fafafa")
    pitch.draw(ax=ax_mini)
    pitch.draw(ax=ax_pc)
    if g.empty or ball_xy is None:
        ax_mini.set_title(f"{source_label}: no data for frame {frame_idx}")
        return fig, False

    teams = _normalise_team(g[team_col])
    xy = g[["x_m", "y_m"]].to_numpy()
    players_mask = teams != -1

    # minimap
    for lbl, colour in TEAM_COLOURS.items():
        m = players_mask & (teams == lbl)
        if m.any():
            ax_mini.scatter(xy[m, 0], xy[m, 1], s=70, c=colour,
                            edgecolors="black", linewidths=0.6, zorder=3)
    ax_mini.scatter([ball_xy[0]], [ball_xy[1]], s=90, c=BALL_COLOUR, marker="*",
                    edgecolors="white", linewidths=0.8, zorder=4)
    ax_mini.set_title(f"Minimap — {source_label}")

    # pitch control
    att, deff, _ = split_attack_defend(xy[players_mask], teams[players_mask], ball_xy)
    pc = pitch_control_surface(att, deff)
    ax_pc.imshow(pc, origin="lower", extent=(0, PITCH_LENGTH_M, 0, PITCH_WIDTH_M),
                 cmap="RdBu_r", vmin=0, vmax=1, alpha=0.65, zorder=2)
    ax_pc.scatter([ball_xy[0]], [ball_xy[1]], s=70, c=BALL_COLOUR, marker="*",
                  edgecolors="white", zorder=5)
    ax_pc.set_title(f"Pitch Control — {source_label}")
    fig.suptitle(f"{clip_id} frame {frame_idx} (SSD-free)", fontsize=12)
    return fig, True


# Representative clips: those present in the committed GT PC table (no SSD needed to pick them).
_gt_pc = pd.read_parquet(OUTPUTS_DIR / "pitch_control_gt_full.parquet")
EXCLUDE_CLIPS = {"SNGS-125", "SNGS-131"}
_clip_pick = (
    _gt_pc[~_gt_pc["clip_id"].isin(EXCLUDE_CLIPS)]
    .sort_values(["action_class", "clip_id"])
    .groupby("action_class").head(1)[["split", "clip_id", "action_class"]]
)

HAS_PIPE_DET = (OUTPUTS_DIR / "detections_soccana_tvcalib.parquet").is_file()
pipe_det = pd.read_parquet(OUTPUTS_DIR / "detections_soccana_tvcalib.parquet") if HAS_PIPE_DET else None

print(f"SSD-free render | clips: {len(_clip_pick)} | pipeline side: {'on' if HAS_PIPE_DET else 'off (parquet absent)'}")
for _, row in _clip_pick.iterrows():
    s, c = row["split"], row["clip_id"]
    frames_here = sorted(_gt_pc[(_gt_pc["split"] == s) & (_gt_pc["clip_id"] == c)]["frame_idx"].unique())
    if not frames_here:
        continue
    f = int(np.median(frames_here))

    # GT side (always available)
    fig, ok = render_minimap_pc(gt_df, "team", s, c, f, "GT")
    if ok:
        out = FIGURES_DIR / f"deploy_minimap_pc_ssd_free_gt_{c}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"  saved {out.name}")
    plt.show(); plt.close(fig)

    # Pipeline side (only when the detections parquet is committed/present)
    if HAS_PIPE_DET:
        fig, ok = render_minimap_pc(pipe_det, "team_kmeans", s, c, f, "Pipeline")
        if ok:
            out = FIGURES_DIR / f"deploy_minimap_pc_ssd_free_pipeline_{c}.png"
            fig.savefig(out, dpi=150, bbox_inches="tight")
            print(f"  saved {out.name}")
        plt.show(); plt.close(fig)